In [48]:
import pandas as pd
import numpy as np
import random
import librosa
import os

from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

from IPython.display import Audio, display
print("done")

done


In [74]:
!pip install soundfile

In [2]:
features_df = pd.read_csv("skrillex2015_features.csv")
transitions_df = pd.read_csv("skrillex2015_transitions.csv")

print(features_df.columns)
print(transitions_df.head())

Index(['Unnamed: 0', 'tempo', 'rms', 'centroid', 'flux', 'mfcc_0', 'mfcc_1',
       'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8',
       'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'track_id'],
      dtype='object')
   Unnamed: 0  from_track  to_track  delta_tempo  delta_rms  delta_centroid  \
0           0           0         1     0.000000  -0.007684      110.187534   
1           1           1         2   -52.615066   0.027054     -223.246427   
2           2           2         3    52.615066  -0.054642      346.407079   
3           3           3         4     0.000000   0.001285       32.823633   
4           4           4         5     0.000000   0.009693      -11.844465   

   delta_flux  delta_mfcc_0  delta_mfcc_1  delta_mfcc_2  delta_mfcc_3  \
0   -0.001972     11.975159      0.822058     -4.808348     -4.341584   
1    0.109363    -14.978062     -1.972160     -2.431379      6.009135   
2   -0.010915     12.155312      2.601774      2.075849    -10.8422

In [3]:
skrillex_cols = [
    'tempo',
    'rms',
    'centroid',
    'flux',
    'mfcc_0',
    'mfcc_1',
    'mfcc_2',
    'mfcc_3',
    'mfcc_4'
]

for col in skrillex_cols:
    features_df[col] = (
        features_df[col]
        .astype(str)
        .str.strip("[]")
        .astype(float)
    )

In [4]:
def clean_value(x):
    # Convert "[123.45]" → 123.45
    if isinstance(x, str):
        x = x.strip("[]")
    return float(x)


def get_track_features(track_id):
    row = features_df[features_df['track_id'] == track_id]
    
    if len(row) == 0:
        return None
    
    vals = row.iloc[0][skrillex_cols]
    
    # Clean every value
    cleaned = [clean_value(v) for v in vals]
    
    return np.array(cleaned)

In [5]:
X = []
y = []

print("Transition columns:", transitions_df.columns.tolist())

# -------------------------------
# Positive transitions (REAL DJ)
# -------------------------------
for _, row in transitions_df.iterrows():
    f1 = get_track_features(row['from_track'])
    f2 = get_track_features(row['to_track'])

    if f1 is not None and f2 is not None:
        diff = np.abs(f1 - f2)
        X.append(diff)
        y.append(1)

print("Positive samples:", len(X))


# -------------------------------
# Negative samples (RANDOM)
# -------------------------------
all_tracks = features_df['track_id'].tolist()

transition_pairs = set(zip(
    transitions_df['from_track'],
    transitions_df['to_track']
))

target_size = len(X) * 2

while len(y) < target_size:
    t1, t2 = random.sample(all_tracks, 2)

    if (t1, t2) not in transition_pairs:
        f1 = get_track_features(t1)
        f2 = get_track_features(t2)

        if f1 is not None and f2 is not None:
            diff = np.abs(f1 - f2)
            X.append(diff)
            y.append(0)

print("Total samples:", len(X))
print("Positive:", sum(y))
print("Negative:", len(y) - sum(y))

Transition columns: ['Unnamed: 0', 'from_track', 'to_track', 'delta_tempo', 'delta_rms', 'delta_centroid', 'delta_flux', 'delta_mfcc_0', 'delta_mfcc_1', 'delta_mfcc_2', 'delta_mfcc_3', 'delta_mfcc_4', 'delta_mfcc_5', 'delta_mfcc_6', 'delta_mfcc_7', 'delta_mfcc_8', 'delta_mfcc_9', 'delta_mfcc_10', 'delta_mfcc_11', 'delta_mfcc_12']
Positive samples: 31
Total samples: 62
Positive: 31
Negative: 31


In [6]:
X = np.array(X)
y = np.array(y)

y = np.array(y).astype(int)
print("Unique labels:", np.unique(y))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

Unique labels: [0 1]


In [7]:
model = MLPClassifier(hidden_layer_sizes=(64, 32), alpha=.01, max_iter=300, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.6923076923076923
              precision    recall  f1-score   support

           0       0.75      0.50      0.60         6
           1       0.67      0.86      0.75         7

    accuracy                           0.69        13
   macro avg       0.71      0.68      0.68        13
weighted avg       0.71      0.69      0.68        13



/opt/anaconda3/envs/dj/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [8]:
audio_folder = "audio"
audio_files = [f for f in os.listdir(audio_folder) if f.endswith(".mp3")]

print(audio_files)

['Pumpupthemind - Electric Flux.mp3', 'Playsound - Play with me.mp3', 'CODESCORN - sawfish.mp3', 'Playsound - The warmth of your soul.mp3', 'TheFrenZ - Against all odds.mp3', 'CXR ATK - Carried by Air.mp3', 'pat102 - fiction.mp3', 'pat102 - space (Handsup Techno).mp3', 'pat102 - skyline.mp3', 'pat102 - rock it.mp3', 'CODESCORN - Dancerfiresgo.mp3', 'CODESCORN - syntetic voice [onlyfans].mp3', 'I_Wanna_Be_With_U_XCX_WORLD_KLICKAUD.mp3', 'pat102 - 145 bpm.mp3', 'Audioinsmusic - Inspired Creativity.mp3.mp3', 'Onedaneira - Flame.mp3.mp3', 'pat102 - 420 (Original).mp3']


In [9]:
def get_phrases(y, sr, duration=5):
    samples_per_phrase = int(sr * duration)
    phrases = []

    for i in range(0, len(y), samples_per_phrase):
        chunk = y[i:i+samples_per_phrase]
        if len(chunk) == samples_per_phrase:
            phrases.append(chunk)

    return phrases

In [10]:
def extract_features(y, sr):
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    rms = np.mean(librosa.feature.rms(y=y))
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    flux = np.mean(librosa.onset.onset_strength(y=y, sr=sr))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=5)
    mfcc_means = np.mean(mfcc, axis=1)

    return np.hstack([tempo, rms, centroid, flux, mfcc_means])

In [11]:
phrase_features = []

for file in audio_files:
    print(f"Processing: {file}")  # simple progress
    
    path = os.path.join(audio_folder, file)
    
    y, sr = librosa.load(path, sr=22050)
    
    phrases = get_phrases(y, sr)

    for p in phrases:
        feat = extract_features(p, sr)
        phrase_features.append((file, feat))

print("Total phrases:", len(phrase_features))

Processing: Pumpupthemind - Electric Flux.mp3
Processing: Playsound - Play with me.mp3
Processing: CODESCORN - sawfish.mp3
Processing: Playsound - The warmth of your soul.mp3
Processing: TheFrenZ - Against all odds.mp3
Processing: CXR ATK - Carried by Air.mp3
Processing: pat102 - fiction.mp3
Processing: pat102 - space (Handsup Techno).mp3
Processing: pat102 - skyline.mp3
Processing: pat102 - rock it.mp3
Processing: CODESCORN - Dancerfiresgo.mp3
Processing: CODESCORN - syntetic voice [onlyfans].mp3
Processing: I_Wanna_Be_With_U_XCX_WORLD_KLICKAUD.mp3
Processing: pat102 - 145 bpm.mp3
Processing: Audioinsmusic - Inspired Creativity.mp3.mp3
Processing: Onedaneira - Flame.mp3.mp3
Processing: pat102 - 420 (Original).mp3
Total phrases: 751


In [12]:
import random

phrase_pairs = []

num_samples = 5000  # 🔥 limit pairs

indices = list(range(len(phrase_features)))

for _ in range(num_samples):
    i, j = random.sample(indices, 2)

    f1 = phrase_features[i][1]
    f2 = phrase_features[j][1]

    diff = np.abs(f1 - f2)
    phrase_pairs.append((i, j, diff))

In [44]:
X_phrases = np.array([p[2] for p in phrase_pairs])
X_phrases_scaled = scaler.transform(X_phrases)

predictions = model.predict(X_phrases_scaled)
probs = model.predict_proba(X_phrases_scaled)[:, 1]
print("Min:", np.min(probs))
print("Max:", np.max(probs))
print("Mean:", np.mean(probs))

Min: 2.1888283881532955e-122
Max: 0.999945141877539
Mean: 0.014429846681266583


In [46]:
results = []

for idx, (i, j, _) in enumerate(phrase_pairs):
    file_i = phrase_features[i][0]
    file_j = phrase_features[j][0]

    if file_i == file_j:
        continue

    score = probs[idx]

    results.append((i, j, score))

results.sort(key=lambda x: x[2], reverse=True)

for i, j, score in results[:10]:
    print(f"{phrase_features[i][0]} → {phrase_features[j][0]} | Score: {score:.3f}")

pat102 - space (Handsup Techno).mp3 → I_Wanna_Be_With_U_XCX_WORLD_KLICKAUD.mp3 | Score: 1.000
CODESCORN - syntetic voice [onlyfans].mp3 → pat102 - 420 (Original).mp3 | Score: 1.000
pat102 - 420 (Original).mp3 → Playsound - Play with me.mp3 | Score: 0.999
pat102 - 420 (Original).mp3 → TheFrenZ - Against all odds.mp3 | Score: 0.999
Onedaneira - Flame.mp3.mp3 → CODESCORN - syntetic voice [onlyfans].mp3 | Score: 0.999
pat102 - rock it.mp3 → pat102 - fiction.mp3 | Score: 0.998
Audioinsmusic - Inspired Creativity.mp3.mp3 → pat102 - space (Handsup Techno).mp3 | Score: 0.998
Playsound - Play with me.mp3 → pat102 - space (Handsup Techno).mp3 | Score: 0.996
CODESCORN - sawfish.mp3 → Pumpupthemind - Electric Flux.mp3 | Score: 0.996
pat102 - fiction.mp3 → pat102 - rock it.mp3 | Score: 0.994


In [36]:
def tempo_baseline(X):
    return (X[:,0] < 10).astype(int)

baseline_preds = tempo_baseline(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, baseline_preds))
print("Baseline F1:", f1_score(y_test, baseline_preds))

Baseline Accuracy: 0.5384615384615384
Baseline F1: 0.7


In [38]:
print("Example feature:", X[0])
print("Label:", y[0])

Example feature: [0.00000000e+00 7.68365000e-03 1.10187534e+02 1.97160000e-03
 1.19751600e+01 8.22058000e-01 4.80834800e+00 4.34158400e+00
 3.32378900e-01]
Label: -1.8626451e-09


In [76]:
import librosa
import numpy as np
import os
import soundfile as sf

def save_transition(file_a, file_b, audio_folder, output_folder, idx, duration=5):

    path_a = os.path.join(audio_folder, file_a)
    path_b = os.path.join(audio_folder, file_b)

    y_a, sr = librosa.load(path_a, sr=22050)
    y_b, sr = librosa.load(path_b, sr=22050)

    # Tempo match
    tempo_a, _ = librosa.beat.beat_track(y=y_a, sr=sr)
    tempo_b, _ = librosa.beat.beat_track(y=y_b, sr=sr)

    if tempo_b > 0:
        rate = float(tempo_a / tempo_b)
        rate = np.clip(rate, 0.5, 2.0)
        y_b = librosa.effects.time_stretch(y_b, rate=rate)

    # Segments
    end_a = y_a[-int(sr * duration):]
    start_b = y_b[:int(sr * duration)]

    # Crossfade
    fade_len = int(sr * 2)

    fade_out = np.linspace(1, 0, fade_len)
    fade_in = np.linspace(0, 1, fade_len)

    end_a[-fade_len:] *= fade_out
    start_b[:fade_len] *= fade_in

    combined = np.concatenate([
        end_a[:-fade_len],
        end_a[-fade_len:] + start_b[:fade_len],
        start_b[fade_len:]
    ])

    # 🔥 Save file
    filename = f"transition_{idx}.wav"
    output_path = os.path.join(output_folder, filename)

    sf.write(output_path, combined, sr)

    print(f"Saved: {filename}")
output_folder = "transitions_output"
os.makedirs(output_folder, exist_ok=True)

In [78]:
for idx, (i, j, score) in enumerate(results[:10]):
    file_a = phrase_features[i][0]
    file_b = phrase_features[j][0]

    print(f"{idx+1}. {file_a} → {file_b} | Score: {score:.3f}")
    
    save_transition(file_a, file_b, audio_folder, output_folder, idx+1)

1. pat102 - space (Handsup Techno).mp3 → I_Wanna_Be_With_U_XCX_WORLD_KLICKAUD.mp3 | Score: 1.000


/var/folders/_4/74l2rxxj7bv53r0y19lt2pw40000gn/T/ipykernel_96542/710287996.py:19: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rate = float(tempo_a / tempo_b)


Saved: transition_1.wav
2. CODESCORN - syntetic voice [onlyfans].mp3 → pat102 - 420 (Original).mp3 | Score: 1.000
Saved: transition_2.wav
3. pat102 - 420 (Original).mp3 → Playsound - Play with me.mp3 | Score: 0.999
Saved: transition_3.wav
4. pat102 - 420 (Original).mp3 → TheFrenZ - Against all odds.mp3 | Score: 0.999
Saved: transition_4.wav
5. Onedaneira - Flame.mp3.mp3 → CODESCORN - syntetic voice [onlyfans].mp3 | Score: 0.999
Saved: transition_5.wav
6. pat102 - rock it.mp3 → pat102 - fiction.mp3 | Score: 0.998
Saved: transition_6.wav
7. Audioinsmusic - Inspired Creativity.mp3.mp3 → pat102 - space (Handsup Techno).mp3 | Score: 0.998
Saved: transition_7.wav
8. Playsound - Play with me.mp3 → pat102 - space (Handsup Techno).mp3 | Score: 0.996
Saved: transition_8.wav
9. CODESCORN - sawfish.mp3 → Pumpupthemind - Electric Flux.mp3 | Score: 0.996
Saved: transition_9.wav
10. pat102 - fiction.mp3 → pat102 - rock it.mp3 | Score: 0.994
Saved: transition_10.wav
